In [1]:
MODEL_CODE = "rf"
MODEL_NAME = "Random Forest"

# 07: Random Forest Modeli

**Proje:** Framingham Kalp Çalışması verisi ile inme (stroke) tahmini · **Model ailesi:** ağaç / bagging

**Random Forest**, çok sayıda Decision Tree'nin oylarını birleştirir (bagging). Her ağaç eğitim verisinden rastgele çekilen farklı bir örneklemle (bootstrap) eğitilir ve her bölmede sütunların yalnızca rastgele bir alt kümesine bakar. Böylece ağaçlar birbirinden farklı hatalar yapar; ortalamaları alındığında tek ağacın iki zayıflığı azalır: ezberleme (overfitting) ve verideki küçük değişikliklere karşı kararsızlık. Bedeli, modelin tek bir ağaç gibi doğrudan okunamamasıdır; kararları SHAP ve LIME ile açıklanır.

**Model neyi kendisi öğrenir, neyi biz seçeriz?** Ağaçların hangi sütunda hangi eşikle bölüneceğini model eğitimde kendisi bulur. Ağaçların ne kadar derinleşeceği, bir yaprakta en az kaç kişi kalacağı, her bölmede kaç sütuna bakılacağı ve dengesizliğin nasıl giderileceği ise eğitimden önce verilen ayarlardır (hiperparametre). Bunlar GridSearch ile, ortak CV katları üzerinde seçilir.

**Bu notebook'ta yapılanlar:** hiperparametre ve dengeleme araması · derinliğin overfitting'e etkisi · feature engineering'in etkisi · final modelin CV değerlendirmesi, eşik seçimi, öğrenme eğrisi · test seti · Decision Tree ile karşılaştırma · XAI (SHAP + LIME)

**Colab'da çalıştırmak için:** Sol menüdeki klasör simgesinden `train.csv`, `test.csv`, `config.py`, `preprocessing.py`, `metrics.py` ve `explain.py` dosyalarını yükleyin. Decision Tree ile karşılaştırma için `07_model_dt` notebook'unun çıktıları (`results.csv`, `dt.joblib`) da gerekir; yoksa o bölüm atlanır.

## 1. Kurulum

In [2]:
# shap and lime are not preinstalled on Colab
try:
    import lime
    import shap
except ImportError:
    %pip install -q shap lime

In [3]:
import sys
from pathlib import Path

# Repo: ../src. Colab: the src/*.py files uploaded to the working directory.
SRC_DIR = next((p for p in (Path("../src"), Path("src"), Path(".")) if (p / "config.py").exists()), None)
if SRC_DIR is None:
    raise FileNotFoundError("config.py bulunamadı. Colab'da src klasöründeki .py dosyalarını sol menüden yükleyin.")
sys.path.insert(0, str(SRC_DIR.resolve()))

# Output folders: models/ and figures/ at the repo root (the working directory on Colab)
ROOT = SRC_DIR.resolve().parent if SRC_DIR.resolve().name == "src" else Path(".")
MODEL_DIR = ROOT / "models";  MODEL_DIR.mkdir(exist_ok=True)
FIG_DIR = ROOT / "figures";   FIG_DIR.mkdir(exist_ok=True)

import time
import warnings

import joblib
import matplotlib.pyplot as plt
import pandas as pd
from imblearn.over_sampling import SMOTE
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV

from config import MAIN_METRIC, RANDOM_STATE, RESULTS_FILE
from explain import explain
from metrics import (SCORING, compare_models, cv_evaluate, learning_curve_analysis, plot_curves,
                     predict_score, shared_cv, test_evaluate, threshold_analysis)
from preprocessing import build_pipeline, load_data

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", None)

## 2. Veri

`train.csv` / `test.csv`. Ön işleme (feature engineering, encoding, ölçekleme, SMOTE) `build_pipeline()` içinde yapılır; test seti en sona kadar kullanılmaz.

In [4]:
X_train, X_test, y_train, y_test = load_data()
print("Eğitim:", X_train.shape, f"| inme: {y_train.sum()} kişi ({y_train.mean():.2%})")
print("Test:  ", X_test.shape, f"| inme: {y_test.sum()} kişi ({y_test.mean():.2%})")

Eğitim: (3521, 17) | inme: 306 kişi (8.69%)
Test:   (881, 17) | inme: 77 kişi (8.74%)


## 3. Hiperparametre ve dengeleme araması

| Parametre | Değerler | Anlamı |
|---|---|---|
| `n_estimators` | 300 (sabit) | Ağaç sayısı. Ağaç eklemek overfitting'e yol açmaz, yalnızca sonucu kararlı hale getirir ve süreyi uzatır; bu yüzden aranmaz |
| `max_depth` | 4, 6, 8, 12, None | Her ağacın en fazla kaç seviye derinleşeceği. `None` sınırsız demektir |
| `min_samples_leaf` | 1, 5, 10, 20 | Bir yaprakta en az kaç kişi kalacağı. Büyük değer, tek tük hastaya özel kural öğrenilmesini engeller |
| `max_features` | sqrt, 0.5 | Her bölmede bakılan sütun oranı. Küçük değer ağaçları birbirinden daha farklı yapar |
| dengeleme | SMOTE, class_weight (balanced), class_weight (balanced_subsample) | SMOTE azınlık sınıfı için sentetik hasta üretir. class_weight veri üretmez, inme vakalarındaki hatayı daha ağır cezalandırır; `balanced_subsample` bu ağırlığı her ağacın kendi örneklemine göre yeniden hesaplar. Ağaçlar SMOTE'un sentetik örneklerini ezberleyebildiği için hepsi denenir |

SMOTE'lu satırlarda pipeline'ın `balance` adımı SMOTE, class_weight'li satırlarda bu adım boştur (`passthrough`). Böylece 120 kombinasyonun tamamı aynı CV katları (`shared_cv()`) ve aynı metriklerle (`SCORING`) karşılaştırılır; en iyi model `MAIN_METRIC` (PR-AUC) ile seçilir.

In [5]:
def balancing_name(balance, class_weight):
    if isinstance(balance, SMOTE) or "SMOTE" in str(balance):
        return "smote"
    return f"class_weight ({class_weight})"


def search_table(search, params):
    """Hiperparametre denemeleri: eğitim ve CV skoru, aradaki fark (overfitting göstergesi)."""
    r = pd.DataFrame(search.cv_results_)
    t = r[[f"param_{p}" for p in params]].map(str)  # None stays as "None"
    t.columns = [p.replace("model__", "") for p in params]
    t.insert(0, "dengeleme", [balancing_name(b, w) for b, w in
                              zip(r["param_balance"], r["param_model__class_weight"])])
    for m in [MAIN_METRIC, "recall", "roc_auc"]:
        t[f"{m}_cv"] = r[f"mean_test_{m}"]
    t[f"{MAIN_METRIC}_train"] = r[f"mean_train_{MAIN_METRIC}"]
    t["fark (train - cv)"] = t[f"{MAIN_METRIC}_train"] - t[f"{MAIN_METRIC}_cv"]
    return t.sort_values(f"{MAIN_METRIC}_cv", ascending=False).round(4)

In [6]:
forest_params = {
    "model__max_depth": [4, 6, 8, 12, None],
    "model__min_samples_leaf": [1, 5, 10, 20],
    "model__max_features": ["sqrt", 0.5],
}
params = [
    {**forest_params, "balance": [SMOTE(random_state=RANDOM_STATE)], "model__class_weight": [None]},
    {**forest_params, "balance": ["passthrough"], "model__class_weight": ["balanced", "balanced_subsample"]},
]

start = time.time()
forest = RandomForestClassifier(n_estimators=300, random_state=RANDOM_STATE, n_jobs=1)
search = GridSearchCV(build_pipeline(forest, balancing="smote"), params, cv=shared_cv(), scoring=SCORING,
                      refit=MAIN_METRIC, n_jobs=-1, return_train_score=True)
search.fit(X_train, y_train)
best = search.best_estimator_   # the pipeline itself, not the search object
best_params = {k.replace("model__", ""): v for k, v in search.best_params_.items() if k.startswith("model__")}
chosen_balancing = balancing_name(search.best_params_["balance"], best_params["class_weight"])

print(f"Süre: {time.time() - start:.0f} sn")
print("Seçilen dengeleme:", chosen_balancing)
print("En iyi parametreler:", best_params)
print(f"En iyi CV {MAIN_METRIC}: {search.best_score_:.4f}")

results = search_table(search, ["model__max_depth", "model__min_samples_leaf", "model__max_features"])
results.head(15)

Süre: 109 sn
Seçilen dengeleme: class_weight (balanced_subsample)
En iyi parametreler: {'class_weight': 'balanced_subsample', 'max_depth': 4, 'max_features': 'sqrt', 'min_samples_leaf': 10}
En iyi CV pr_auc: 0.2088


,dengeleme,max_depth,min_samples_leaf,max_features,pr_auc_cv,recall_cv,roc_auc_cv,pr_auc_train,fark (train - cv)
82,class_weight (balanced_subsample),4,10,sqrt,0.2088,0.5229,0.7124,0.3148,0.1060
51,class_weight (balanced),6,20,sqrt,0.2075,0.5556,0.7146,0.3750,0.1675
83,class_weight (balanced_subsample),4,20,sqrt,0.2072,0.5295,0.7119,0.2948,0.0876
41,class_weight (balanced),4,5,sqrt,0.2062,0.5818,0.7140,0.3326,0.1264
40,class_weight (balanced),4,1,sqrt,0.2059,0.5818,0.7126,0.3395,0.1336
43,class_weight (balanced),4,20,sqrt,0.2055,0.5948,0.7135,0.2901,0.0847
86,class_weight (balanced_subsample),4,10,0.5,0.2055,0.5587,0.7165,0.3268,0.1213
47,class_weight (balanced),4,20,0.5,0.2053,0.6439,0.7164,0.3079,0.1026
42,class_weight (balanced),4,10,sqrt,0.2052,0.5915,0.7132,0.3133,0.1081
46,class_weight (balanced),4,10,0.5,0.2043,0.6308,0.7165,0.3247,0.1205


### Derinliğin etkisi (overfitting)

Her derinlik için, her dengeleme yönteminde en iyi `min_samples_leaf` / `max_features` seçilerek eğitim ve CV skorları çizilir. Derinlik arttıkça eğitim skoru yükselir. CV skoru bir noktadan sonra yükselmiyor ya da düşüyorsa ağaçlar ezberlemeye başlamıştır. Decision Tree'nin aynı grafiğiyle karşılaştırıldığında, ağaçların ortalamasını almanın CV skorundaki düşüşü ne kadar azalttığı görülür.

In [7]:
r = results.copy()
r["max_depth"] = r["max_depth"].replace("None", "sınırsız")
order = ["4", "6", "8", "12", "sınırsız"]
colors = {"smote": "#4C72B0", "class_weight (balanced)": "#DD8452",
          "class_weight (balanced_subsample)": "#55A868"}

fig, ax = plt.subplots(figsize=(8, 4.5))
for bal, color in colors.items():
    sub = (r[r["dengeleme"] == bal].sort_values(f"{MAIN_METRIC}_cv", ascending=False)
           .drop_duplicates("max_depth").set_index("max_depth").loc[order])
    ax.plot(order, sub[f"{MAIN_METRIC}_train"], "o--", color=color, label=f"eğitim ({bal})")
    ax.plot(order, sub[f"{MAIN_METRIC}_cv"], "o-", color=color, label=f"CV ({bal})")
ax.set_xlabel("max_depth")
ax.set_ylabel(MAIN_METRIC)
ax.set_title("Random Forest: derinliğin etkisi")
ax.legend(fontsize=8)
plt.tight_layout()
plt.savefig(FIG_DIR / f"{MODEL_CODE}_depth_overfitting.png", dpi=200)
plt.show()

## 4. Feature engineering'in etkisi

Ortak pipeline 5 yeni özellik ekler (17 → 28 sütun). Türetilen özelliklerin katkı sağlayıp sağlamadığı en iyi parametrelerle kontrol edilir (bu deneyler `results.csv`'ye yazılmaz).

In [8]:
rows = {}
for fe in [True, False]:
    pipe = build_pipeline(forest, balancing="smote", feature_engineering=fe)
    pipe.set_params(**search.best_params_)
    res = cv_evaluate(pipe, X_train, y_train, model_name=f"{MODEL_NAME} (FE={fe})", save=False, verbose=False)
    rows["FE açık" if fe else "FE kapalı"] = res["summary"]["cv_mean"]
pd.DataFrame(rows).T.round(3)

metric,recall,precision,f1,specificity,roc_auc,pr_auc,accuracy
FE açık,0.523,0.160,0.245,0.739,0.712,0.209,0.720
FE kapalı,0.542,0.165,0.253,0.740,0.718,0.198,0.723


## 5. Final model: CV değerlendirmesi

Seçilen model ortak `cv_evaluate` ile değerlendirilir ve sonucu `results.csv`'ye **bir satır** olarak eklenir. Karşılaştırma notebook'u bu satırı kullanır. Eğitim ve CV skorları arasındaki fark (`gap`) overfitting göstergesidir.

In [9]:
cv_result = cv_evaluate(best, X_train, y_train, model_name=MODEL_NAME,
                        note=f"{chosen_balancing} + RF, GridSearch: {best_params}")

===== Random Forest: 5 katlı CV (eşik = 0.5) =====
Kat başına ortalama eğitim süresi: 0.36 sn

             train_mean  cv_mean  cv_std    gap
metric                                         
recall            0.672    0.523   0.061  0.149
precision         0.203    0.160   0.014  0.043
f1                0.312    0.245   0.022  0.067
specificity       0.748    0.739   0.012  0.009
roc_auc           0.797    0.712   0.018  0.085
pr_auc            0.315    0.209   0.027  0.106
accuracy          0.742    0.720   0.009  0.021

Uyarı: pr_auc için eğitim - CV farkı 0.106. Overfitting olabilir.

Sonuç 'results.csv' dosyasına eklendi.


## 6. Eşik seçimi

Varsayılan eşik 0.5'tir, ama inmeyi kaçırmak yanlış alarmdan pahalı olduğu için eşik, **test setine bakılmadan** CV tahminleri (OOF) üzerinde seçilir: hedef recall'u (%80) en az yanlış alarmla sağlayan eşik.

In [10]:
table, threshold = threshold_analysis(cv_result["y"], cv_result["oof_score"], model_name=MODEL_NAME)

===== Random Forest: Eşik analizi (toplam 306 inme, 3521 kişi) =====
 threshold  caught  missed  false_alarms  recall  precision  specificity  false_alarms_per_1000
      0.05     306       0          3215   1.000      0.087        0.000                913.093
      0.10     306       0          3215   1.000      0.087        0.000                913.093
      0.15     306       0          3203   1.000      0.087        0.004                909.685
      0.20     302       4          3032   0.987      0.091        0.057                861.119
      0.25     292      14          2666   0.954      0.099        0.171                757.171
      0.30     281      25          2209   0.918      0.113        0.313                627.379
      0.35     263      43          1834   0.859      0.125        0.430                520.875
      0.40     240      66          1524   0.784      0.136        0.526                432.832
      0.45     198     108          1140   0.647      0.148        

## 7. Overfitting kontrolü: öğrenme eğrisi

Eğitim verisi büyüdükçe eğitim ve CV skorları birbirine yaklaşıyorsa model genelleşiyor demektir. Büyük ve kapanmayan bir fark overfitting, ikisinin birlikte düşük kalması underfitting gösterir.

In [11]:
lc = learning_curve_analysis(best, X_train, y_train, model_name=MODEL_NAME)
lc.round(3)

,train_size,train_mean,cv_mean,cv_std
0,281,0.707,0.163,0.027
1,788,0.502,0.191,0.032
2,1295,0.416,0.189,0.024
3,1802,0.364,0.196,0.016
4,2309,0.343,0.199,0.023
5,2816,0.315,0.202,0.024


## 8. Test seti (yalnızca bir kez)

Final model tüm eğitim verisiyle yeniden eğitilir, seçilen eşikle test setinde değerlendirilir ve `models/` klasörüne kaydedilir.

In [12]:
best.fit(X_train, y_train)
test_result = test_evaluate(best, X_test, y_test, threshold=threshold,
                            model_name=MODEL_NAME, cv_result=cv_result)

joblib.dump(best, MODEL_DIR / f"{MODEL_CODE}.joblib")
print(f"Model kaydedildi: models/{MODEL_CODE}.joblib")

===== Random Forest: Test seti (eşik = 0.389) =====
67/77 inme yakalandı, 10 inme kaçırıldı, 399 yanlış alarm (881 kişi).

             cv_same_threshold   test  cv_minus_test
recall                   0.801  0.870         -0.069
precision                0.133  0.144         -0.010
f1                       0.229  0.247         -0.018
specificity              0.505  0.504          0.001
roc_auc                  0.710  0.763         -0.053
pr_auc                   0.189  0.243         -0.054
accuracy                 0.531  0.536         -0.005
Model kaydedildi: models/rf.joblib


## 9. Decision Tree ile karşılaştırma

Random Forest, Decision Tree'lerin birleşimi olduğu için ailenin asıl sorusu şudur: ağaçların ortalamasını almak tek ağaca göre ne kazandırıyor? CV sonuçları `results.csv`'den, test eğrileri iki kayıtlı modelden çizilir. `07_model_dt` çalıştırılmadıysa bu bölüm atlanır.

In [13]:
dt_path = MODEL_DIR / "dt.joblib"
if dt_path.exists() and Path(RESULTS_FILE).exists():
    cv_table = compare_models(plot=False)
    display(cv_table.loc[[m for m in ["Decision Tree", MODEL_NAME] if m in cv_table.index]].round(3))

    dt = joblib.load(dt_path)
    plot_curves({"Decision Tree": (y_test, predict_score(dt, X_test)),
                 MODEL_NAME: (y_test, test_result["score"])}, title="Test seti")
    plt.gcf().savefig(FIG_DIR / "dt_rf_test_curves.png", dpi=200, bbox_inches="tight")
else:
    print("Decision Tree çıktıları bulunamadı, karşılaştırma atlandı.")

===== Model karşılaştırması (CV ortalaması ± std, pr_auc'e göre sıralı) =====
                      recall      precision             f1    specificity        roc_auc         pr_auc      accuracy  pr_auc_train_cv_gap  fit_time_s
model_name                                                                                                                                            
Random Forest  0.523 ± 0.061   0.16 ± 0.014  0.245 ± 0.022  0.739 ± 0.012  0.712 ± 0.018  0.209 ± 0.027  0.72 ± 0.009                0.106        0.36
Decision Tree   0.546 ± 0.09  0.147 ± 0.021   0.23 ± 0.022  0.692 ± 0.071  0.662 ± 0.026  0.168 ± 0.045  0.68 ± 0.058                0.022        0.02


,recall,precision,f1,specificity,roc_auc,pr_auc,accuracy
model_name,,,,,,,
Decision Tree,0.546,0.147,0.230,0.692,0.662,0.168,0.68
Random Forest,0.523,0.160,0.245,0.739,0.712,0.209,0.72


## 10. XAI: modelin açıklanması

Random Forest için ortak `explain()` fonksiyonu **TreeExplainer** kullanır: tüm test verisi açıklanır (300 ağaç olduğu için birkaç dakika sürebilir). Fonksiyona aramanın kendisi değil, `best` (pipeline) verilir.

**Grafik okuma:** Bar grafiğinde çubuk uzunluğu önemi gösterir. Özet grafiğinde kırmızı = özelliğin değeri yüksek, mavi = düşük; nokta sağdaysa inme riskini artırıyor. LIME'da yeşil çubuklar "inme var" kararını destekleyen koşullardır.

In [14]:
# explain() shows its plots in order: SHAP bar, SHAP summary, LIME. Each one is also saved to figures/.
_names = iter([f"{MODEL_CODE}_shap_bar.png", f"{MODEL_CODE}_shap_summary.png", f"{MODEL_CODE}_lime.png"])
_show = plt.show
def _save_and_show(*args, **kwargs):
    name = next(_names, None)
    if name:
        plt.gcf().savefig(FIG_DIR / name, dpi=200, bbox_inches="tight")
    _show(*args, **kwargs)

plt.show = _save_and_show
try:
    start = time.time()
    xai = explain(best, X_train, X_test, model_name=MODEL_NAME)
    print(f"\nXAI süresi: {time.time() - start:.0f} sn")
finally:
    plt.show = _show

===== Random Forest: XAI Analizi =====



En önemli 5 özellik (SHAP):
               feature  mean_abs_shap
                   AGE       0.062312
     comorbidity_count       0.025662
                 SYSBP       0.022806
mean_arterial_pressure       0.019731
               PREVHYP       0.019214

[Random Forest] Test hastası #438: İnme olasılığı = 76.93%

XAI süresi: 1 sn


In [15]:
xai["shap_importance"].head(10)

,feature,mean_abs_shap
0,AGE,0.062312
1,comorbidity_count,0.025662
2,SYSBP,0.022806
3,mean_arterial_pressure,0.019731
4,PREVHYP,0.019214
5,pulse_pressure,0.014438
6,DIABP,0.013430
7,bp_stage_stage2,0.011019
8,HEARTRTE,0.009737
9,CIGPDAY,0.006713


## 11. Özet (rapor için)

### Neden Random Forest?

- **Tek ağacın zayıflıklarına çözüm:** Decision Tree okunabilir ama tek başına sınırlı kaldı (CV PR-AUC 0,168, ROC-AUC 0,66) ve verideki küçük değişikliklere duyarlıdır. Random Forest, farklı örneklemlerle ve her bölmede farklı sütun alt kümeleriyle eğitilen yüzlerce ağacın oylarını birleştirir. Ağaçların hataları birbirinden farklı olduğu için ortalamaları hem daha isabetli hem daha kararlıdır.
- **Tablo verisinde güçlü bir yöntem:** Ölçekleme gerektirmez, sayısal ve ikili sütunları birlikte kullanır, özellikler arasındaki etkileşimleri kendiliğinden yakalar. Bu tür sağlık verilerinde sık kullanılan, güvenilir bir karşılaştırma modelidir.
- **Açıklanabilirlik:** Tek ağaç gibi çizilerek okunamaz, ancak ağaç tabanlı olduğu için SHAP'ın hızlı ve kesin `TreeExplainer` yöntemiyle tüm test verisi açıklanabilir.

### Yöntem

- Ağaç sayısı 300'de sabitlendi (ağaç eklemek overfitting'e yol açmaz, yalnızca sonucu kararlı hale getirir). `max_depth`, `min_samples_leaf`, `max_features` ve dengeleme yöntemi (SMOTE, `class_weight="balanced"`, `class_weight="balanced_subsample"`) 120 kombinasyonluk bir GridSearch ile, ekibin ortak 5 katlı CV bölmesi üzerinde arandı. En iyi model PR-AUC ile seçildi.
- Seçilen model: **derinlik 4, yaprak başına en az 10 kişi, her bölmede sütunların karekökü kadarına bakılması (`sqrt`), `class_weight="balanced_subsample"`**. İlk 15 kombinasyonun tamamı `class_weight` kullanan modellerdir: Random Forest'ta, sentetik hasta üretmek yerine inme vakalarına daha fazla ağırlık vermek daha iyi sonuç verdi.
- Sığ ağaçlar (derinlik 4) en iyi sonucu verdi. Derin ağaçlarda eğitim skoru yükselirken CV skoru artmamaktadır (`rf_depth_overfitting.png`).
- Feature engineering'in etkisi küçüktür: türetilen özelliklerle CV PR-AUC 0,209, onlarsız 0,198; diğer metrikler neredeyse aynıdır. Ekip pipeline'ı ile tutarlılık için özellikler korundu.

### Sonuçlar

| | CV (5 kat, eşik 0,5) | CV (seçilen eşik 0,39) | Test (881 kişi, eşik 0,39) |
|---|---|---|---|
| Recall | 0,523 | 0,801 | 0,870 |
| Precision | 0,160 | 0,133 | 0,144 |
| Specificity | 0,739 | 0,505 | 0,504 |
| ROC-AUC | 0,712 | | 0,763 |
| PR-AUC | 0,209 | | 0,243 |

- Eşik, test setine bakılmadan CV tahminleri üzerinde, vakaların en az %80'ini en az yanlış alarmla yakalayacak şekilde 0,39 seçildi.
- Test setinde 77 inme vakasının **67'si yakalandı, 10'u kaçırıldı**; 804 sağlıklı kişiden 399'una yanlış alarm verildi.
- Eğitim PR-AUC'si (0,315) CV'den (0,209) yüksektir. Ancak öğrenme eğrisinde fark veri arttıkça hızla kapanmaktadır (eğitim 0,71 → 0,32, CV 0,16 → 0,20) ve test skorları CV'den düşük değildir; model yeni veriye genellenmektedir. Ağaç topluluklarında eğitim skorunun CV'den bir miktar yüksek olması beklenen bir durumdur.

### Decision Tree ile karşılaştırma

| | Decision Tree | Random Forest |
|---|---|---|
| CV PR-AUC | 0,168 ± 0,045 | **0,209 ± 0,027** |
| CV ROC-AUC | 0,662 ± 0,026 | **0,712 ± 0,018** |
| Test PR-AUC | 0,158 | **0,243** |
| Test ROC-AUC | 0,698 | **0,763** |
| Test: yakalanan / kaçırılan vaka | 68 / 9 | 67 / 10 |
| Test: yanlış alarm | 483 | **399** |

- Random Forest, Decision Tree'ye göre bütün eşikten bağımsız metriklerde daha iyidir. Aynı düzeyde vaka yakalarken (%87–88) **84 daha az yanlış alarm** verir.
- Katlar arası standart sapma da daha küçüktür (PR-AUC 0,027 / 0,045): ağaçların ortalamasını almak modeli daha kararlı hale getirmiştir. Bu, bagging'in beklenen iki etkisini (daha iyi genelleme, daha düşük varyans) doğrular.

### Model neye dayanıyor?

- SHAP'a göre en önemli özellikler: **yaş** (açık ara), **eşlik eden hastalık sayısı**, **sistolik tansiyon**, ortalama arter basıncı ve hipertansiyon öyküsü. Tansiyonla ilgili birçok sütun (SYSBP, ortalama arter basıncı, nabız basıncı, DIABP, tansiyon evresi) listede yer alır; bu sütunlar birbirinden türetildiği için tansiyonun toplam etkisi bunlara bölünmüştür ve tek tek göründüğünden daha büyüktür.
- Decision Tree'de de ilk iki özellik yaş ve eşlik eden hastalık sayısıydı. İki modelin aynı risk faktörlerini öne çıkarması ve bunların bilinen inme risk faktörleri olması, sonuçların güvenilirliğini destekler.

### Değerlendirme

- Random Forest, ailenin en iyi modelidir. Yine de inmeyi bu ölçümlerle tahmin etmek zordur: vakaların %87'sini yakalamak için riskli denilen her 7 kişiden yalnızca 1'i gerçekten inme geçirmiştir. Model bir tanı aracı değil, ileri tetkik gerektiren kişileri belirleyen bir tarama aracı olarak yorumlanmalıdır.